# Core Reranker
vLLM-Hook is an extensible framework that aims to allow selective access to model internals during the inference. 
As a demonstration of that, in this notebook, we show how vLLM-Hook enables *Core Reranker* for document relevance scoring. 

**Paper**: [Contrastive Retrieval Heads Improve Attention-Based Re-Ranking](https://arxiv.org/abs/2510.02219).<br />
**Authors**: Linh Tran, Yulong Li, Radu Florian, Wei Sun <br />
**"TL;DR"**: Core reranker is an attention-based reranker that leverage attention weights from selected transformer heads to produce document relevance scores.


### Installation
If running this from a new environment, please use the cell below to install `vllm_hook_plugins`. Update the path/command to match your environment.<br />
The following block is not necessary if running this notebook from an environment where the package has already been installed.

In [1]:
from pathlib import Path
import sys

# vllm_hooks/notebooks/
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent

PKG_DIR = REPO_ROOT/"vllm_hook_plugins"
REQ_FILE = REPO_ROOT/"requirement.txt"

print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root   :", REPO_ROOT)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

%pip install -e "{PKG_DIR}"

if REQ_FILE.exists():
    %pip install -r "{REQ_FILE}"
else:
    print("⚠️ requirements.txt not found at", REQ_FILE)


Notebook dir: /Users/timothyburley/opensource/vLLM-Hook/notebooks/metal
Repo root   : /Users/timothyburley/opensource/vLLM-Hook
Package dir : /Users/timothyburley/opensource/vLLM-Hook/vllm_hook_plugins
Req file    : /Users/timothyburley/opensource/vLLM-Hook/requirement.txt
Obtaining file:///Users/timothyburley/opensource/vLLM-Hook/vllm_hook_plugins
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vllm-hook-plugins (pyproject.toml) ... done
  Created wheel for vllm-hook-plugins: filename=vllm_hook_plugins-0.2.0-0.editable-py3-none-any.whl size=3147 sha256=6f176899ccca7e962ae6aab8023fde2bcbb49c03eae26691a84efaeb02240b69
  Stored in directory: /private/var/folders/dn/99pbhj4d48n4r8rg_hvqtglr0000gn/T/pip-ephem-wheel-cache-kp3pon3j/wheels/91/fa/cf/bacb8fa72ad781d6b97e1ba762fa3be0ed4d9aa39201e4b56d
Successfully 

### Importing the Hook-Enabled LLM
The plugin provides its own LLM wrapper that behaves like vllm.LLM (`from vllm import LLM`) but adds support for hooks and instrumentation.
We import it here:

In [2]:
from vllm_hook_plugins.metal import HookLLMMetal

INFO 06-07 17:04:34 [__init__.py:44] Available plugins for group vllm.platform_plugins:
INFO 06-07 17:04:34 [__init__.py:46] - metal -> vllm_metal:register
INFO 06-07 17:04:34 [__init__.py:49] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 06-07 17:04:37 [__init__.py:238] Platform plugin metal is activated
INFO 06-07 17:04:38 [importing.py:69] Triton not installed or not compatible; certain GPU-related functions will not be available.


### Environment & multiprocessing setup

In [3]:
import os
import multiprocessing as mp
import torch
from typing import List
mp.set_start_method("spawn", force=True)
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

### Helper functions that give the instruction range
As Core Reranker needs to locate the candidate passages and the user query in the prompt, below is a helper function that gives the data range with texts.<br />
Check [Core Reranker](https://arxiv.org/pdf/2510.02219) for more details.

In [4]:
def apply_chat_template_and_get_ranges(tokenizer, model_name: str, query: str, documents: List[str]):
    # setup prompts
    off_set = 0
    if 'granite' in model_name.lower():
        prompt_prefix = '<|start_of_role|>user<|end_of_role|>'
        prompt_suffix = '<|end_of_text|><|start_of_role|>assistant<|end_of_role|>'
    elif 'llama' in model_name.lower():
        prompt_prefix = '<|start_header_id|>user<|end_header_id|>'
        prompt_suffix = '<|eot_id|><|start_header_id|>assistant<|end_header_id|>'
    elif 'mistral' in model_name.lower():
        prompt_prefix = '[INST]'
        prompt_suffix = '[/INST]'
        off_set = 1
    elif 'phi' in model_name.lower():
        prompt_prefix = '<|im_start|>user<|im_sep|>'
        prompt_suffix = '<|im_end|><|im_start|>assistant<|im_sep|>'
    retrieval_instruction = ' Here are some paragraphs:\n\n'
    retrieval_instruction_late = 'Please find information that are relevant to the following query in the paragraphs above.\n\nQuery: '
    
    doc_span = []
    query_start_idx = None
    query_end_idx = None

    llm_prompt = prompt_prefix + retrieval_instruction

    for i, doc in enumerate(documents):

        llm_prompt += f'[document {i+1}]'
        start_len = len(tokenizer(llm_prompt).input_ids)

        llm_prompt += ' ' + " ".join(doc)
        end_len = len(tokenizer(llm_prompt).input_ids) - off_set

        doc_span.append((start_len, end_len))
        llm_prompt += '\n\n'

    start_len = len(tokenizer(llm_prompt).input_ids)

    llm_prompt += retrieval_instruction_late
    after_retrieval_instruction_late = len(tokenizer(llm_prompt).input_ids) - off_set

    llm_prompt += f'{query.strip()}'
    end_len = len(tokenizer(llm_prompt).input_ids) - off_set
    llm_prompt += prompt_suffix

    query_start_idx = start_len
    query_end_idx = end_len

    return llm_prompt, (doc_span, query_start_idx, after_retrieval_instruction_late, query_end_idx)

### Initialize `HookLLMMetal`
Before we create the LLM instance, we need to specify the model and data type:

In [5]:
cache_dir = '~/.cache'  # Specify cache dir
model = 'mistralai/Mistral-7B-Instruct-v0.3' 
    
dtype_map = {
    'mistralai/Mistral-7B-Instruct-v0.3': torch.float16,
}

We also need to provide a config file that specifies the important heads we want to track. <br />
For Core Reranker, this config file can be obtained from [head_detection.py](https://github.com/linhhtran/CoRe-Reranking/blob/main/experiments/head_detection.py). 

In [6]:
import json
from pathlib import Path

json_path = Path("../../model_configs/core_reranker/Mistral-7B-Instruct-v0.3.json")  # adjust path

with open(json_path, "r") as f:
    config = json.load(f)

# print(config)

Inside `probe_hook_qk` and `core_reranker` we defined the desired behavior during model inference and after the model inference: 
- `workers/metal/probe_hookqk_worker_metal.py` defines that we need `q` (query) and `k` (key) to be saved during forward passes
- `analyzers/metal/core_reranker_analyzer_metal.py` calculates the passage relevance score and the final ranking of passages

Now, we initialize the llm:

In [ ]:
llm = HookLLMMetal(
    model=model,
    worker_name="probe_hook_qk",
    analyzer_name="core_reranker",
    config_file=json_path,
    download_dir=cache_dir,
    gpu_memory_utilization=0.7,
    trust_remote_code=True,
    dtype=dtype_map[model],
    enable_prefix_caching=True,
    max_model_len=2048,
    enable_hook=True
)

HookLLMMetal worker=probe_hook_qk hooks_enabled=True
INFO 06-07 17:04:39 [utils.py:278] non-default args: {'trust_remote_code': True, 'download_dir': '~/.cache', 'dtype': torch.float16, 'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 2048, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
WARNING 06-07 17:04:39 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_WORKER
WARNING 06-07 17:04:39 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_RECLAIM_BASE_FOR_ENCODE
WARNING 06-07 17:04:39 [arg_utils.py:1552] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 06-07 17:04:40 [config.py:316] Inferred from consolidated*.safetensors files torch.bfloat16 dtype.
INFO 06-07 17:04:40 [model.py:617] Resolved architecture: MistralForCausa

Multiple valid tokenizer files found. Using tokenizer.model.v3.


INFO 06-07 17:04:42 [core.py:112] Initializing a V1 LLM engine (v0.22.0) with config: model='mistralai/Mistral-7B-Instruct-v0.3', speculative_config=None, tokenizer='mistralai/Mistral-7B-Instruct-v0.3', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=2048, download_dir='~/.cache', load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=True, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cpu, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_e

mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

INFO 06-07 17:04:52 [model_lifecycle.py:187] Model loaded in 9.62s: mistralai/Mistral-7B-Instruct-v0.3
INFO 06-07 17:04:52 [cache_policy.py:680] MLX path: reporting 0.27 GB for scheduler admission control (one max-length sequence, max_model_len=2048)
INFO 06-07 17:04:52 [kv_cache_utils.py:1733] GPU KV cache size: 2,048 tokens
INFO 06-07 17:04:52 [kv_cache_utils.py:1734] Maximum concurrency for 2,048 tokens per request: 1.00x
INFO 06-07 17:04:52 [cache_policy.py:297] KV cache config received: 128 blocks (MLX manages cache internally)
INFO 06-07 17:04:52 [model_runner.py:649] Warming up model...
INFO 06-07 17:04:57 [model_runner.py:655] Model warm-up complete
INFO 06-07 17:04:57 [core.py:302] init engine (profile, create kv cache, warmup model) took 5.15 s (compilation: 5.12 s)


### Test case
In the following, we show a test case with seven candidate passages and a user query.

In [8]:
case = {
        "query": "Which came first, the invention of the telephone or the light bulb?",
        "documents": [
            [
            "Alexander Graham Bell is credited with inventing the first practical telephone.",
            " He was awarded the U.S. patent for the invention of the telephone on March 7, 1876.",
            " The first successful demonstration of the telephone took place shortly thereafter, when Bell famously called his assistant, saying, 'Mr. Watson, come here, I want to see you.'",
            " Bell’s invention revolutionized communication by allowing people to talk to each other over long distances."
            ],
            [
            "Thomas Edison is widely known for inventing the first commercially practical incandescent light bulb.",
            " Although he did not invent the concept of the light bulb itself, Edison developed a version that was safe, affordable, and long-lasting.",
            " His patent for the electric light bulb was filed in 1879, three years after Bell’s telephone patent.",
            " Edison's innovation led to widespread use of electric lighting and helped usher in the modern electrical age."
            ],
            [
            "Before Edison, several inventors worked on early versions of the light bulb.",
            " Sir Humphry Davy created the first electric arc lamp in the early 1800s, and later inventors like Joseph Swan in Britain improved upon the design.",
            " However, these early bulbs were inefficient or burned out quickly, and it was Edison who perfected the design for everyday use."
            ],
            [
            "The telephone was invented before the practical light bulb.",
            " Bell’s patent for the telephone was issued in 1876, while Edison’s patent for the light bulb was filed in 1879.",
            " Thus, the telephone came first."
            ],
            [
            "Both the telephone and the light bulb are considered groundbreaking inventions of the late 19th century.",
            " The telephone transformed communication, while the light bulb transformed how people lived and worked at night.",
            " Together, they symbolize the rapid technological progress of that era."
            ],
            [
            "Edison and Bell were contemporaries and pioneers of the Second Industrial Revolution.",
            " Their inventions marked major milestones in human history, driving the growth of telecommunications and electrical infrastructure."
            ],
            [
            "In summary, the telephone was invented in 1876 and the light bulb in 1879.",
            " Therefore, the invention of the telephone came first."
            ]
        ]
    }

Next, we apply chat template and obtain the input range using the helper function defined above.<br />
Specifically, as core reranker relies on the aggregated attentions from the user query to each passage, it needs a reference attention baseline for each passage. The authors swap the user query with `'N/A'` and treat the resulting aggregated attention as the normalizing factor for each passage.

In [9]:
query = case["query"]
documents = case["documents"]
        
# Apply chat template and get ranges
query_text, query_spec = apply_chat_template_and_get_ranges(llm.tokenizer, model, query, documents)
na_text, na_spec = apply_chat_template_and_get_ranges(llm.tokenizer, model, 'N/A', documents)

Finally, we perform the model inference:

In [10]:
llm.generate(query_text, temperature=0.1, max_tokens=1)
llm.generate(na_text, cleanup=False, temperature=0.1, max_tokens=1)

Releasing base engine before Metal encode-hook capture.


INFO 06-07 17:04:58 [utils.py:278] non-default args: {'trust_remote_code': True, 'download_dir': '~/.cache', 'dtype': torch.float16, 'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 2048, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
WARNING 06-07 17:04:58 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_WORKER
WARNING 06-07 17:04:58 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_RECLAIM_BASE_FOR_ENCODE
WARNING 06-07 17:04:58 [arg_utils.py:1552] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 06-07 17:04:59 [model.py:617] Resolved architecture: MistralForCausalLM
WARNING 06-07 17:04:59 [model.py:2090] Casting torch.bfloat16 to torch.float16.
INFO 06-07 17:04:59 [model.py:1752] Using max model len 2048
WARNING 06-0

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.05s/it, est. speed input: 285.20 toks/s, output: 0.49 toks/s]

INFO 06-07 17:05:02 [utils.py:278] non-default args: {'trust_remote_code': True, 'download_dir': '~/.cache', 'dtype': torch.float16, 'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 2048, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
WARNING 06-07 17:05:02 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_WORKER
WARNING 06-07 17:05:02 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_RECLAIM_BASE_FOR_ENCODE
WARNING 06-07 17:05:02 [arg_utils.py:1552] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


...

INFO 06-07 17:05:08 [utils.py:278] non-default args: {'trust_remote_code': True, 'download_dir': '~/.cache', 'dtype': torch.float16, 'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 2048, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
WARNING 06-07 17:05:08 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_WORKER
WARNING 06-07 17:05:08 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_RECLAIM_BASE_FOR_ENCODE


WARNING 06-07 17:05:09 [arg_utils.py:1552] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 06-07 17:05:09 [model.py:617] Resolved architecture: MistralForCausalLM
WARNING 06-07 17:05:09 [model.py:2090] Casting torch.bfloat16 to torch.float16.
INFO 06-07 17:05:09 [model.py:1752] Using max model len 2048
WARNING 06-07 17:05:09 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-07 17:05:09 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-07 17:05:09 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
INFO 06-07 17:05:09 [platform.py:286] Metal: disabled chunked prefill

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.53s/it, est. speed input: 376.06 toks/s, output: 0.66 toks/s]


[RequestOutput(request_id=0, prompt="[INST] Here are some paragraphs:\n\n[document 1] Alexander Graham Bell is credited with inventing the first practical telephone.  He was awarded the U.S. patent for the invention of the telephone on March 7, 1876.  The first successful demonstration of the telephone took place shortly thereafter, when Bell famously called his assistant, saying, 'Mr. Watson, come here, I want to see you.'  Bell’s invention revolutionized communication by allowing people to talk to each other over long distances.\n\n[document 2] Thomas Edison is widely known for inventing the first commercially practical incandescent light bulb.  Although he did not invent the concept of the light bulb itself, Edison developed a version that was safe, affordable, and long-lasting.  His patent for the electric light bulb was filed in 1879, three years after Bell’s telephone patent.  Edison's innovation led to widespread use of electric lighting and helped usher in the modern electrical

During the model inference in the previous step, vLLM-Hook has automatically saved selected queries and keys. Now, we can directly call the analyzer to get the passage relevance score and the final ranking of passages:

In [11]:

stats = llm.analyze(analyzer_spec={'query_spec': query_spec, 'na_spec': na_spec})

Finally we can print out the results as follows:

In [12]:
print(f"Sorted document IDs and scores by CoRe-Reranking: {stats['ranking']}: {stats['scores']}")

Sorted document IDs and scores by CoRe-Reranking: [[6, 3, 1, 0, 4, 5, 2]]: [[5.6875, 3.953125, 2.3125, 2.078125, 1.5625, 0.734375, 0.55078125]]
